In [ ]:
import polars as pl
import polars.selectors as cs
from project_paths import paths

data = paths.processed_data / "unified_aims_eir_bgs.parquet"
df = pl.read_parquet(data)


In [ ]:

halcrow_types = {
    "Embankment": "embankment",
    "Wall": "vertical_wall",
    "Flood Gate": "flood_gate",
    "Demountable Defence": "demountable", 
    "Beach": "beach", 
    "Barrier Beach": "beach",
    "Dunes": "dunes",
    "Weir": "weir",
    "Engineered High Ground": "embankment", 
    "Natural High Ground": "embankment",
}

tiers = {
    "Embankment": "direct",
    "Wall": "direct",
    "Flood Gate": "direct",
    "Demountable Defence": "direct",
    "Beach": "direct",
    "Barrier Beach": "direct",
    "Dunes": "direct",
    "Weir": "direct",
    # keeping high ground and using the embankment curve, but carry original type so they can be filtered out for validation
    "Engineered High Ground": "proxy_embankment",
    "Natural High Ground": "proxy_embankment",
}


In [ ]:

mapping = pl.DataFrame(
    {
        "aims__asset_sub_type": list(halcrow_types.keys()),
        "halcrow_type": list(halcrow_types.values()),
        "tier": [tiers[k] for k in halcrow_types],
    }
)

mapping


In [ ]:


environ = {
    "Fluvial": "fluvial", 
    "Coastal": "coastal", 
    "Tidal": "coastal", 
    "Fluvial/Tidal": "coastal"
}

grade = (
    pl.col("eir__condition_grade").cast(pl.Utf8).str.strip_chars().str.extract(r"^(\d+)").cast(pl.Int8)
)


df_scoped = (
    df
    .with_columns(grade.alias("grade"))
    .filter(pl.col("grade").is_between(1, 5))
    .join(mapping, on="aims__asset_sub_type", how="left")
    .with_columns(
        pl.col("tier").fill_null("unmapped"),
        pl.col("aims__protection_type").replace_strict(environ, default="fluvial").alias("environment"),
        pl.when(pl.col("halcrow_type") == "vertical_wall").then(pl.lit("concrete_masonry"))
            .when(pl.col("halcrow_type") == "flood_gate").then(pl.lit("metal"))
            .when(pl.col("halcrow_type") == "embankment").then(pl.lit("varying_core"))
            .otherwise(pl.lit("na")).alias("material"),
        pl.when(pl.col("halcrow_type") == "embankment").then(pl.lit("narrow"))
            .otherwise(pl.lit("na")).alias("width"),
    )
    .rename({"aims__asset_sub_type": "original_sub_type"})
)


df_scoped




In [ ]:


mapped = df_scoped.filter(pl.col("tier") != "unmapped")

mapped 



In [ ]:

print("coverage of graded set by Halcrow mappability")

print(
    df_scoped.group_by("tier").agg(pl.len().alias("n"))
    .with_columns((pl.col("n") / pl.col("n").sum() * 100).round(1).alias("pct"))
    .sort("n", descending=True)
)

print()
print("mapped rows by original type")
print(mapped.group_by("original_sub_type", "halcrow_type", "tier").agg(pl.len()).sort("len", descending=True))

check start date coverage

In [ ]:

def to_date(col):
    return (
        pl.col(col).str.strptime(pl.Date, "%d/%m/%Y", strict=False) if mapped.schema[col] == pl.Utf8 else pl.col(col).cast(pl.Date)
    )


In [ ]:

assert "aims__asset_start_date" in mapped.columns


In [ ]:

df_aged = mapped.with_columns(
    to_date("aims__asset_start_date").alias("start_date"),
    to_date("eir__inspection_date").alias("asof_date"),
).with_columns(
    ((pl.col("asof_date") - pl.col("start_date")).dt.total_days() / 365.25).alias("age_years")
)

df_aged


In [ ]:

df_aged = df_aged.with_columns(
    ((pl.col("age_years") >= 0) & (pl.col("age_years") <= 150)).alias("age_valid")
)

df_aged


In [ ]:

df_aged_tiers_validation = df_aged.group_by("tier").agg(
    pl.len().alias("n"),
    pl.col("start_date").is_not_null().mean().mul(100).round(1).alias("has_start_date_pct"),
    pl.col("age_valid").mean().mul(100).round(1).alias("usable_age_pct"),
).sort("n", descending=True)

print(df_aged_tiers_validation)




In [ ]:

df_aged_spikes = df_aged.group_by(pl.col("start_date").dt.year().alias("start_year")).agg(pl.len()).sort("len", descending=True).head(15)

df_aged_spikes



In [ ]:


frame_aged = df_aged.filter(pl.col("age_valid")) # rows the curve can be run on
print(frame_aged.height, mapped.height)

apply the halcrow deterioration curves. 

using the tables in the SC060078 guidance report.

In [ ]:

# key = (halcrow_type, environment, material, width)
# value = {(rate, regime): [t1, t2, t3, t4, t5]}


curves = {
    # embankment, varying core (pg. 75 fluvial, pg. 77 coastal), narrow
    ("embankment", "fluvial", "varying_core", "narrow"): {
        ("slowest", 1): [0, 5, 10, 40, 60], 
        ("slowest", 2): [0, 20, 40, 70, 110], 
        ("slowest", 3): [0, 22, 44, 90, 130],
        ("medium", 1): [0, 3, 6, 25, 40],   
        ("medium", 2): [0, 15, 30, 60, 80],   
        ("medium", 3): [0, 16, 33, 70, 90],
        ("fastest", 1): [0, 1, 3, 5, 7],    
        ("fastest", 2): [0, 2, 5, 7, 10],     
        ("fastest", 3): [0, 3, 6, 8, 11],
    },
    ("embankment", "coastal", "varying_core", "narrow"): {
        ("slowest", 1): [0, 5, 10, 40, 60],
        ("slowest", 2): [0, 20, 40, 60, 80],
        ("slowest", 3): [0, 22, 45, 80, 110],
        ("medium", 1): [0, 3, 6, 22, 30],
        ("medium", 2): [0, 14, 28, 40, 50],
        ("medium", 3): [0, 15, 30, 45, 60],
        ("fastest", 1): [0, 1, 2, 4, 5],
        ("fastest", 2): [0, 2, 4, 6, 8],
        ("fastest", 3): [0, 3, 5, 8, 10],
    },

    # vertical wall, concrete/brick-masonry (from Table 2.3)
    ("vertical_wall", "fluvial", "concrete_masonry", "na"): {
        ("slowest", 1): [0, 20, 50, 70, 80],
        ("slowest", 2): [0, 25, 60, 100, 120],
        ("slowest", 3): [0, 30, 70, 130, 160],
        ("medium", 1): [0, 15, 35, 50, 60],
        ("medium", 2): [0, 20, 45, 70, 90],
        ("medium", 3): [0, 25, 55, 90, 120],
        ("fastest", 1): [0, 5, 20, 30, 40],
        ("fastest", 2): [0, 10, 30, 50, 60],
        ("fastest", 3): [0, 15, 40, 70, 80],
    },
    ("vertical_wall", "coastal", "concrete_masonry", "na"): {
        ("slowest", 1): [0, 15, 45, 60, 80], 
        ("slowest", 2): [0, 20, 60, 80, 100], 
        ("slowest", 3): [0, 25, 75, 100, 120],
        ("medium", 1): [0, 10, 30, 40, 50],  
        ("medium", 2): [0, 15, 40, 55, 70],   
        ("medium", 3): [0, 20, 50, 70, 90],
        ("fastest", 1): [0, 5, 15, 25, 30],  
        ("fastest", 2): [0, 10, 20, 30, 40],  
        ("fastest", 3): [0, 15, 25, 35, 50],
    },

    # vertical wall, timber (from pg. 48,49) 
    ("vertical_wall", "fluvial", "timber", "na"): {
        ("slowest", 1): [0, 7, 15, 18, 21],
        ("slowest", 2): [0, 15, 30, 35, 40],
        ("slowest", 3): [0, 23, 45, 52, 60],
        ("medium", 1): [0, 5, 10, 12, 15],
        ("medium", 2): [0, 10, 20, 25, 30],
        ("medium", 3): [0, 15, 30, 35, 42],
        ("fastest", 1): [0, 3, 5, 7, 10],
        ("fastest", 2): [0, 5, 10, 12, 15],
        ("fastest", 3): [0, 7, 15, 17, 20],
    },
    ("vertical_wall", "coastal", "timber", "na"): {
        ("slowest", 1): [0, 5, 13, 16, 20], 
        ("slowest", 2): [0, 14, 28, 33, 38], 
        ("slowest", 3): [0, 21, 42, 48, 55],
        ("medium", 1): [0, 4, 8, 10, 14],   
        ("medium", 2): [0, 8, 18, 23, 28],   
        ("medium", 3): [0, 13, 28, 33, 38],
        ("fastest", 1): [0, 2, 4, 6, 8],    
        ("fastest", 2): [0, 4, 8, 10, 13],   
        ("fastest", 3): [0, 5, 13, 15, 18],
    },

    # flood gates and barriers, metal (from pg. 196 197)
    ("flood_gate", "fluvial", "metal", "na"): {
        ("slowest", 1): [0, 15, 32, 41, 50], 
        ("slowest", 2): [0, 20, 40, 50, 60], 
        ("slowest", 3): [0, 25, 48, 59, 70],
        ("medium", 1): [0, 12, 25, 32, 38],  
        ("medium", 2): [0, 18, 34, 42, 50],  
        ("medium", 3): [0, 24, 43, 52, 62],
        ("fastest", 1): [0, 5, 12, 16, 20],  
        ("fastest", 2): [0, 10, 22, 30, 35], 
        ("fastest", 3): [0, 15, 32, 44, 50],
    },
    ("flood_gate", "coastal", "metal", "na"): {
        ("slowest", 1): [0, 13, 22, 26, 30], 
        ("slowest", 2): [0, 18, 29, 35, 40], 
        ("slowest", 3): [0, 23, 36, 44, 50],
        ("medium", 1): [0, 10, 14, 16, 18],  
        ("medium", 2): [0, 15, 23, 27, 30],  
        ("medium", 3): [0, 20, 32, 38, 42],
        ("fastest", 1): [0, 4, 7, 9, 10],    
        ("fastest", 2): [0, 7, 11, 13, 15],  
        ("fastest", 3): [0, 10, 15, 17, 20],
    },


    # demountable defence (on pg. 67, tbl C.4 metal/wood)
    ("demountable_defence", "fluvial", "metal", "na"): {
        ("slowest", 1): [0, 2, 4, 5, 7],
        ("slowest", 2): [0, 10, 20, 60, 70],
        ("slowest", 3): [0, 15, 25, 70, 80],
        ("medium", 1): [0, 1, 3, 4, 5],
        ("medium", 2): [0, 5, 10, 45, 55],
        ("medium", 3): [0, 8, 15, 55, 65],
        ("fastest", 1): [0, 1, 2, 3, 4],
        ("fastest", 2): [0, 2, 5, 35, 45],
        ("fastest", 3): [0, 5, 10, 45, 55],
    },
    ("demountable_defence", "fluvial", "wood", "na"): {
        ("slowest", 1): [0, 2, 4, 5, 7],
        ("slowest", 2): [0, 5, 10, 30, 35],
        ("slowest", 3): [0, 8, 13, 35, 40],
        ("medium", 1): [0, 1, 3, 4, 5],
        ("medium", 2): [0, 3, 5, 23, 28],
        ("medium", 3): [0, 4, 8, 28, 33],
        ("fastest", 1): [0, 1, 2, 3, 4],
        ("fastest", 2): [0, 1, 3, 18, 23],
        ("fastest", 3): [0, 3, 5, 23, 28],
    },

    #  beach (C.7 pg 130)
    ("beach", "coastal", "shingle/sand", "na"): {
        ("slowest", 1): [0, 15, 38, 75, 100],
        ("slowest", 2): [0, 27, 50, 150, 200],
        ("slowest", 3): [0, 27, 75, 200, 250],
        ("medium", 1): [0, 9, 13, 25, 35],
        ("medium", 2): [0, 16, 30, 50, 75 ],
        ("medium", 3): [0, 20, 55, 90, 120],
        ("highest", 1): [0, 4, 7, 9, 13],
        ("highest", 2): [0, 7, 10, 13, 20],
        ("highest", 3): [0, 12, 20, 25, 40],
    },

    # dunes (C.15 pg 152)
    ("dunes", "coastal", "", ""): {
        ("slowest", 1): [0, 20, 40, 110, 150],
        ("slowest", 2): [0, 27, 60, 150, 200],
        ("slowest", 3): [0, 30, 80, 190, 250],
        ("medium", 1): [0, 10, 15, 30, 40],
        ("medium", 2): [0, 15, 35, 60, 80],
        ("medium", 3): [0,20, 60, 100, 130],
        ("highest", 1): [0, 5, 8, 10, 15],
        ("highest", 2): [0, 7, 10, 13, 20],
        ("highest", 3): [0, 12, 20, 25, 40],
    },

    # weir (C.18 pg. 167)
    ("weir", "fluvial", "all", "na"): {
        ("slowest", 1): [0, 20, 30, 50, 70],
        ("slowest", 2): [0, 40, 70, 90, 110],
        ("slowest", 3): [0, 60, 110, 130, 150],
        ("medium", 1): [0, 15, 20, 40, 60],
        ("medium", 2): [0, 30, 50, 70, 90],
        ("medium", 3): [0, 45, 80, 100, 120],
        ("highest", 1): [0, 10, 15, 30, 40],
        ("highest", 2): [0, 20, 30, 50, 60],
        ("highest", 3): [0, 30, 45, 70, 80],
    },

}


rows = []
for (htype, env, mat, width), grid in curves.items():

    for (rate, regime), t in grid.items():
        rows.append(
            {
                "halcrow_type": htype, 
                "environment": env, 
                "material": mat, 
                "width": width,
                "rate": rate, 
                "regime": regime,
                "t1": t[0],
                "t2": t[1],
                "t3": t[2],
                "t4": t[3],
                "t5": t[4],
            }
        )



curve_df = pl.DataFrame(rows)

curve_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(len(curves.keys()), 3, figsize=(16,40), sharex=True, sharey=True)


for i, ((htype, env, mat, width), grid) in enumerate(curves.items()):
    ax_col = axes[i]


    rows = []
    for (rate, regime), t in grid.items():

        for i, t in enumerate(t):
            rows.append(
                {
                    "rate": rate, 
                    "regime": regime,
                    "t": t,
                    "y": 5 - i,
                }
            )



    data = pl.DataFrame(rows)
    # data = pl.DataFrame(rows).unpivot(on=cs.numeric(), index=["regime", "rate"])


    for i, regime in enumerate(data.get_column("regime").unique().to_list()):
        ax = ax_col[i]

        subset = data.filter(pl.col("regime").eq(regime))

        # sns.lineplot(data=subset, x="t", y="y", hue="regime", style="rate", ax=ax)
        sns.lineplot(data=subset, x="t", y="y", style="rate", ax=ax)
        ax.set_title(f"{htype}, {env}, {mat}, regime {regime}")


plt.show()

    

In [ ]:
# with no date, im chosing a medium maintainance regime to apply to all assets. 
# from the report:
    # Three maintenance regimes have been considered, namely: 
    # Maintenance Regime 1: Low/basic – do minimum repair/maintenance 
    #  Inspection + H&S repair (annually) 
    # Maintenance Regime 2: Medium maintenance regime 
    #  Inspection + H&S repair (annually) 
    #  Maintenance activities as proposed in the Environment Agency Maintenance 
    # Standards (Environment Agency 2010 and Environment Agency 2012, Appendix 
    # B) for maintaining at target CG 3 (Note: The maintenance standards will also 
    # pick up minor reactive repairs) 
    # Maintenance Regime 3: High maintenance regime 
    #  Inspection + H&S repair (annually) 
    #  Maintenance activities as proposed in the Environment Agency Maintenance 
    # Standards for maintaining at target CG 2 (Note: The maintenance standards will 
    # also pick up minor reactive repairs) 
#
# its possible that we can infer the maintainance regime from the target condition? seems that this says that regime 3 is needed to maintain at condition grade 2, so if target grade is 2 then regime is 3...



rate, regime = "medium", 2


In [ ]:

def frac(low, high):
    low_e, high_e = pl.col(low), pl.col(high)
    return pl.when(high_e > low_e).then((pl.col("age_years") - low_e) / (high_e - low_e)).otherwise(0.0)





In [ ]:

prediction = (
    frame_aged
    .join(
        curve_df.filter((pl.col("rate").eq(rate)) & (pl.col("regime").eq(regime))),
        on=["halcrow_type", "environment", "material", "width"], 
        how="left",
    )
    .with_columns(
        pl.when(pl.col("t2").is_null()).then(pl.lit("unimplemented"))
            .otherwise(pl.lit("scored")).alias("curve_status")
    )
    .with_columns(
        pl.when(pl.col("age_years").ge(pl.col("t5"))).then(5.0)
            .when(pl.col("age_years").ge(pl.col("t4"))).then(4.0 + frac("t4", "t5"))
            .when(pl.col("age_years").ge(pl.col("t3"))).then(3.0 + frac("t3", "t4"))
            .when(pl.col("age_years").ge(pl.col("t2"))).then(2.0 + frac("t2", "t3"))
            .otherwise(1.0 + frac("t1", "t2"))
        .clip(1.0, 5.0)
        .alias("pred_grade_cont")
    )
    .with_columns(pl.col("pred_grade_cont").floor().cast(pl.Int8).clip(1, 5).alias("pred_grade"))
)


prediction



In [ ]:


scored = prediction.filter(pl.col("curve_status").eq("scored"))

scored


In [ ]:
print(scored.height)

print(prediction.height - scored.height)



In [ ]:

comparison_grade = scored.group_by(["grade", "pred_grade"]).agg(pl.len()).sort(["grade", "pred_grade"]).select(
    (pl.col("pred_grade") - pl.col("grade")).abs().mean().alias("mae"),
    # (pl.col("pred_grade_cont") - pl.col("grade")).abs().mean().alias("mae_cont"),
)


comparison_grade_cont = scored.group_by(["grade", "pred_grade_cont"]).agg(pl.len()).sort(["grade", "pred_grade_cont"]).select(
    # (pl.col("pred_grade") - pl.col("grade")).abs().mean().alias("mae"),
    (pl.col("pred_grade_cont") - pl.col("grade")).abs().mean().alias("mae_cont"),
)




In [ ]:
comparison_grade

In [ ]:
comparison_grade_cont

In [ ]:
scored.get_column("original_sub_type").unique()

In [ ]:
scored_exclude_nhg = scored.filter(pl.col("original_sub_type").ne("Natural High Ground"))

comparison_grade_ex_nhg = scored_exclude_nhg.group_by(["grade", "pred_grade"]).agg(pl.len()).sort(["grade", "pred_grade"]).select(
    (pl.col("pred_grade") - pl.col("grade")).abs().mean().alias("mae"),
    # (pl.col("pred_grade_cont") - pl.col("grade")).abs().mean().alias("mae_cont"),
)

comparison_grade_ex_nhg

In [ ]:
scored_exlude_all_hg = scored.filter(~(pl.col("original_sub_type").is_in(["Natural High Ground", "Engineered High Ground"])))

comparison_grade_ex_all_hg = scored_exlude_all_hg.group_by(["grade", "pred_grade"]).agg(pl.len()).sort(["grade", "pred_grade"]).select(
    (pl.col("pred_grade") - pl.col("grade")).abs().mean().alias("mae"),
    # (pl.col("pred_grade_cont") - pl.col("grade")).abs().mean().alias("mae_cont"),
)

comparison_grade_ex_all_hg

even after fixing curve direction bug, mae is still the same when excluding assets. i think this is because the mae calc itself is wrong

In [ ]:

_test_comparison_grade = scored.select(
    (pl.col("pred_grade") - pl.col("grade")).alias("error")
)

_test_comparison_grade




In [ ]:
_test_comparison_grade = _test_comparison_grade.with_columns(
    pl.col("error").abs().alias("absolute_error")
)

_test_comparison_grade

In [ ]:
_test_comparison_grade = _test_comparison_grade.select(
    pl.col("absolute_error").mean().alias("mae")
)

_test_comparison_grade

In [ ]:

comparison_grade = scored.select(
    (pl.col("pred_grade") - pl.col("grade")).abs().mean().alias("mae"),
)

comparison_grade_cont = scored.select(
    (pl.col("pred_grade_cont") - pl.col("grade")).abs().mean().alias("mae_cont"),
)

print(comparison_grade, "\n\n", comparison_grade_cont)

In [ ]:
comparison_grade_ex_nhg = scored_exclude_nhg.select(
    (pl.col("pred_grade") - pl.col("grade")).abs().mean().alias("mae"),
)

comparison_grade_cont_ex_nhg = scored_exclude_nhg.select(
    (pl.col("pred_grade_cont") - pl.col("grade")).abs().mean().alias("mae_cont"),
)

print(comparison_grade_ex_nhg, "\n\n", comparison_grade_cont_ex_nhg)

In [ ]:
comparison_grade_ex_all_hg = scored_exlude_all_hg.select(
    (pl.col("pred_grade") - pl.col("grade")).abs().mean().alias("mae"),
)

comparison_grade_cont_ex_all_hg = scored_exlude_all_hg.select(
    (pl.col("pred_grade_cont") - pl.col("grade")).abs().mean().alias("mae_cont"),
)

print(comparison_grade_ex_all_hg, "\n\n", comparison_grade_cont_ex_all_hg)

Im really suprised that mae is this high, its saying that on average an asset is a grade out. I wonder what the mae is if you assign grade randomly or apply 3 flatly across everything. This is pretty good evidence that there ought to be a better tool than the current heuristics. 

In [ ]:
import numpy as np

heuristic_vs_naive = scored.select(
    pl.col("grade"),
    pl.col("pred_grade")
).with_columns(
    pl.lit(value=3).alias("flat_baseline"),
    pl.Series(values=[np.random.randint(low=1, high=6) for _ in range(scored.height)]).alias("random_baseline")
)

heuristic_vs_naive

In [ ]:
heuristic_vs_naive_comp = heuristic_vs_naive.select(
    (pl.col("pred_grade") - pl.col("grade")).abs().mean().alias("mae"),
    (pl.col("flat_baseline") - pl.col("grade")).abs().mean().alias("flt_base_mae"),
    (pl.col("random_baseline") - pl.col("grade")).abs().mean().alias("rand_base_mae"),
)

heuristic_vs_naive_comp

so it looks like the heuristic is about as good as random and a lot worse than applying a flat median value.

next I need to look for patterns. For example, is it always under predicting, does it affect certain types of assets etc.

In [ ]:
prediction_tab = scored.pivot(on="pred_grade", index="grade", values="pred_grade", aggregate_function="len", sort_columns=True).sort("grade")

prediction_tab 


columns are predictions, rows are actuals



In [ ]:

scored.select((pl.col("pred_grade") - pl.col("grade")).mean().alias("direction_of_bias"))

slightly over predicting, but compared to size of absolute error its a fairly random distribution of errors.

In [ ]:
type_comp = scored.select(
    pl.col("grade"),
    pl.col("pred_grade"),
    pl.col("original_sub_type")
).with_columns(
    pl.lit(value=3).alias("flat_baseline"),
    pl.Series(values=[np.random.randint(low=1, high=6) for _ in range(scored.height)]).alias("random_baseline")
).with_columns(
    (pl.col("pred_grade") - pl.col("grade")).abs().alias("absolute_error"),
    (pl.col("flat_baseline") - pl.col("grade")).abs().alias("flt_base_absolute_error"),
    (pl.col("random_baseline") - pl.col("grade")).abs().alias("rand_base_absolute_error"),
).group_by("original_sub_type").mean()


type_comp


Wall is the only type with underprediction, all others over predict. so perhaps errors not so symetrically distributed. 

These findings are pretty interesting - heuristic is not useful with this dataset. But the "with this dataset" bit is pretty important, theres lots of assumptions going into this:
    - guessing the maintainance regime
    - guessing the material
    - always using the medium rate

in particular the material is important. An actual engineer using the heuristics is likely to have access to detailed information about the singular asset they are assessing. It only doing the full national analysis that im doing with the aims dataset in particular that this information is missing. This reframes the heuristic performance. 

I can say that inputing the median is better than using the heuristic with imputed defaults, but not that median impution is better than these heuristics overall.

the median imput benchmark will also have to be one i carry forward to evaluate ml models. Theres obviously the big caveate that it will never predict grades 4 and 5 which are the ones that actually need predicting so repair can be schedules, performance decrease can be mitigated, etc. But it is a higher benchmark and one that an ML model probably has to beat or be comparible to plus have all the grades in order to be considered useful. 

